# Statistical Tests for Time Series

Beyond stationarity testing, several statistical tests are routinely used in
time series analysis for model validation and causal inference.  This notebook
covers:

1. **Ljung-Box test** — are residual autocorrelations collectively zero?
2. **Granger causality test** — do past values of X help predict Y?
3. **Cointegration test** — do two non-stationary series share a common trend?

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import grangercausalitytests, coint, adfuller
from statsmodels.graphics.tsaplots import plot_acf

# Plotting defaults
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

In [ ]:
# --- Load datasets ---
# Airline Passengers
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

# Daily Female Births (stationary)
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

# M2 Money Stock
m2 = pd.read_csv(
    DATA_DIR / "M2SLMoneyStock.csv",
    index_col="DATE",
    parse_dates=True,
)
m2.index.freq = "MS"
m2.columns = ["M2"]

# PCE Personal Spending
pce = pd.read_csv(
    DATA_DIR / "PCEPersonalSpending.csv",
    index_col="DATE",
    parse_dates=True,
)
pce.index.freq = "MS"
pce.columns = ["PCE"]

print("M2 Money Stock:", m2.shape)
print("PCE Spending:  ", pce.shape)

---
## 1. Ljung-Box Test

### Purpose

The **Ljung-Box test** checks whether a set of autocorrelations are
**significantly different from zero** as a group.  It is most commonly used to
test whether model **residuals** are white noise.

### Hypotheses

- $H_0$: The data are **independently distributed** (no autocorrelation up to
  lag $m$)
- $H_1$: The data exhibit **serial correlation** at one or more lags

### Test Statistic

$$
Q = T(T+2) \sum_{k=1}^{m} \frac{r_k^2}{T - k}
$$

Under $H_0$, $Q \sim \chi^2(m - p - q)$ where $p$ and $q$ are the orders of
the fitted ARMA model (0 for raw data).

### Interpretation

- **p-value > 0.05** $\Rightarrow$ fail to reject $H_0$ $\Rightarrow$ no
  significant autocorrelation $\Rightarrow$ residuals look like white noise
- **p-value $\le$ 0.05** $\Rightarrow$ reject $H_0$ $\Rightarrow$ significant
  autocorrelation remains $\Rightarrow$ model is inadequate

In [ ]:
# Example 1: white noise — should PASS (high p-values)
np.random.seed(42)
white_noise = pd.Series(np.random.randn(500))

lb_wn = acorr_ljungbox(white_noise, lags=[5, 10, 15, 20], return_df=True)

print("Ljung-Box Test — White Noise")
print(lb_wn.to_string())
print()
print("All p-values are large — fail to reject H0 — no autocorrelation detected.")

In [ ]:
# Example 2: airline passengers (raw) — should FAIL (low p-values)
lb_airline = acorr_ljungbox(airline["Passengers"], lags=[5, 10, 15, 20], return_df=True)

print("Ljung-Box Test — Airline Passengers (raw)")
print(lb_airline.to_string())
print()
print("All p-values are essentially zero — strong autocorrelation.")
print("This is expected: the raw series is not white noise.")

In [ ]:
# Example 3: Daily Female Births — some autocorrelation expected
lb_births = acorr_ljungbox(births["Births"], lags=[5, 10, 15, 20], return_df=True)

print("Ljung-Box Test — Daily Female Births")
print(lb_births.to_string())
print()
print("Even a stationary series can have autocorrelation —")
print("stationarity != white noise.")

In [ ]:
# Visualise: ACF of white noise vs correlated data side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_acf(white_noise, lags=20, ax=axes[0], title="ACF — White Noise")
plot_acf(births["Births"], lags=20, ax=axes[1], title="ACF — Daily Births")

plt.tight_layout()
plt.show()

print("The Ljung-Box test formalises what we see visually:")
print("white noise has all spikes within the bands; births has several outside.")

### Ljung-Box at multiple lags

It is good practice to test at several lag values.  A common approach is to
test at lag $m = \min(10, T/5)$ and also at the seasonal period (e.g., 12 for
monthly data).

In [ ]:
# Full lag-by-lag Ljung-Box for white noise
lb_full = acorr_ljungbox(white_noise, lags=20, return_df=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lb_full.index, lb_full["lb_pvalue"], color="steelblue", alpha=0.7)
ax.axhline(0.05, color="red", linestyle="--", label="$\\alpha = 0.05$")
ax.set_xlabel("Lag")
ax.set_ylabel("p-value")
ax.set_title("Ljung-Box p-values by Lag — White Noise")
ax.legend()
plt.tight_layout()
plt.show()

print("All p-values are above the 0.05 line — consistent with white noise.")

---
## 2. Granger Causality Test

### Purpose

The **Granger causality test** assesses whether past values of one time series
$X$ contain useful information for predicting another series $Y$, beyond what
is already contained in $Y$'s own past.

### Hypotheses

- $H_0$: Lagged values of $X$ do **not** help predict $Y$ (no Granger
  causality)
- $H_1$: Lagged values of $X$ **do** help predict $Y$

### Important Caveat

> **Granger causality does NOT imply true causation.** It only tests
> predictive usefulness.  The name is unfortunate — it should really be called
> "Granger predictability".

### Requirements

- Both series should be **stationary** (apply differencing first if needed)
- The test is **directional**: $X$ Granger-causing $Y$ is different from $Y$
  Granger-causing $X$

In [ ]:
# Align M2 and PCE to a common date range
common_idx = m2.index.intersection(pce.index)
m2_aligned = m2.loc[common_idx, "M2"]
pce_aligned = pce.loc[common_idx, "PCE"]

print(f"Common date range: {common_idx[0].date()} to {common_idx[-1].date()}")
print(f"Number of observations: {len(common_idx)}")

# Plot both series
fig, ax1 = plt.subplots(figsize=(12, 5))

color1, color2 = "tab:blue", "tab:orange"
ax1.plot(m2_aligned, color=color1, label="M2 Money Stock")
ax1.set_ylabel("M2 (Billions $)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(pce_aligned, color=color2, label="PCE Personal Spending")
ax2.set_ylabel("PCE (Billions $)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

ax1.set_title("M2 Money Stock vs PCE Personal Spending")
fig.legend(loc="upper left", bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# Both series are non-stationary — difference first
m2_diff = m2_aligned.diff().dropna()
pce_diff = pce_aligned.diff().dropna()

# Verify stationarity
m2_adf = adfuller(m2_diff, autolag="AIC")[1]
pce_adf = adfuller(pce_diff, autolag="AIC")[1]

print(f"ADF p-value (M2 diff):  {m2_adf:.6f} — {'Stationary' if m2_adf < 0.05 else 'Non-stationary'}")
print(f"ADF p-value (PCE diff): {pce_adf:.6f} — {'Stationary' if pce_adf < 0.05 else 'Non-stationary'}")

In [ ]:
# Granger causality: does M2 Granger-cause PCE?
# grangercausalitytests expects a DataFrame with columns [Y, X]
granger_df = pd.DataFrame({
    "PCE": pce_diff,
    "M2": m2_diff,
}).dropna()

print("=" * 60)
print("Test: Does M2 Granger-cause PCE?")
print("=" * 60)
gc_result = grangercausalitytests(granger_df[["PCE", "M2"]], maxlag=4, verbose=True)

In [ ]:
# Reverse direction: does PCE Granger-cause M2?
print("=" * 60)
print("Test: Does PCE Granger-cause M2?")
print("=" * 60)
gc_result_rev = grangercausalitytests(granger_df[["M2", "PCE"]], maxlag=4, verbose=True)

In [ ]:
# Summarise Granger results
print("Granger Causality Summary (F-test p-values):")
print("="*50)
print(f"{'Lag':<6} {'M2 -> PCE':<15} {'PCE -> M2':<15}")
print("-"*50)

for lag in range(1, 5):
    p_m2_pce = gc_result[lag][0]["ssr_ftest"][1]
    p_pce_m2 = gc_result_rev[lag][0]["ssr_ftest"][1]
    print(f"{lag:<6} {p_m2_pce:<15.6f} {p_pce_m2:<15.6f}")

print()
print("Interpretation: examine which directions have p < 0.05.")

### Negative example — unrelated series

In [ ]:
# Granger causality between M2 and a random series — should NOT be significant
np.random.seed(99)
random_series = pd.Series(
    np.random.randn(len(m2_diff)),
    index=m2_diff.index,
    name="Random",
)

granger_neg = pd.DataFrame({"Random": random_series, "M2": m2_diff}).dropna()

print("=" * 60)
print("Negative control: Does M2 Granger-cause a random series?")
print("=" * 60)
gc_neg = grangercausalitytests(granger_neg[["Random", "M2"]], maxlag=4, verbose=True)

In [ ]:
print("As expected, no significant Granger causality with random data.")
print("This confirms the test is working correctly — it does not produce")
print("false positives on unrelated series.")

---
## 3. Cointegration Test

### Purpose

Two non-stationary series $X_t$ and $Y_t$ are **cointegrated** if there
exists a linear combination $Y_t - \beta X_t$ that is stationary.  In other
words, the series share a common stochastic trend — they may wander apart in
the short run, but a long-run equilibrium pulls them back together.

### Why It Matters

- If two series are cointegrated, a VAR model in differences is
  **mis-specified** — a **Vector Error Correction Model (VECM)** is needed
  instead.
- Cointegration is the basis for **pairs trading** in finance.
- We will revisit this concept in Chapter 10 (VAR models).

### Hypotheses (Engle-Granger test)

- $H_0$: No cointegration
- $H_1$: Series are cointegrated

In [ ]:
# Test for cointegration between M2 and PCE (in levels, not differenced)
coint_stat, coint_pvalue, coint_crit = coint(m2_aligned, pce_aligned)

print("Engle-Granger Cointegration Test: M2 vs PCE")
print(f"  Test Statistic: {coint_stat:.6f}")
print(f"  p-value:        {coint_pvalue:.6f}")
print(f"  Critical Values (1%, 5%, 10%): {coint_crit}")
print()
if coint_pvalue < 0.05:
    print("  => Reject H0: the series ARE cointegrated.")
else:
    print("  => Fail to reject H0: no evidence of cointegration.")

In [ ]:
# Test cointegration with an obviously unrelated series
np.random.seed(42)
random_walk = pd.Series(
    np.cumsum(np.random.randn(len(m2_aligned))),
    index=m2_aligned.index,
)

coint_stat2, coint_pvalue2, coint_crit2 = coint(m2_aligned, random_walk)

print("Cointegration Test: M2 vs Random Walk")
print(f"  Test Statistic: {coint_stat2:.6f}")
print(f"  p-value:        {coint_pvalue2:.6f}")
print()
if coint_pvalue2 < 0.05:
    print("  => Evidence of cointegration (surprising).")
else:
    print("  => No cointegration (expected — unrelated series).")

In [ ]:
# Visualise: cointegrated vs non-cointegrated pairs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# M2 vs PCE (potentially cointegrated)
ax1 = axes[0]
ax1.plot(m2_aligned / m2_aligned.iloc[0], label="M2 (normalised)")
ax1.plot(pce_aligned / pce_aligned.iloc[0], label="PCE (normalised)", alpha=0.8)
ax1.set_title("M2 vs PCE (normalised)")
ax1.legend()

# M2 vs random walk (not cointegrated)
ax2 = axes[1]
ax2.plot(m2_aligned / m2_aligned.iloc[0], label="M2 (normalised)")
rw_norm = (random_walk - random_walk.iloc[0]) / random_walk.std() + 1
ax2.plot(rw_norm, label="Random Walk (scaled)", alpha=0.8)
ax2.set_title("M2 vs Random Walk")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 4. Choosing the Right Test

| Question | Test | Key Idea |
|----------|------|----------|
| Is the series stationary? | ADF, KPSS | Unit root / stationarity tests (Notebook 02) |
| Are residuals white noise? | **Ljung-Box** | Tests if autocorrelations are collectively zero |
| Does X predict Y? | **Granger causality** | Predictive usefulness (NOT true causation) |
| Do X and Y share a trend? | **Cointegration** | Linear combination is stationary |

In [ ]:
# Quick reference: common null hypotheses
tests = pd.DataFrame({
    "Test": ["ADF", "KPSS", "Ljung-Box", "Granger", "Engle-Granger Coint."],
    "H0 (null)": [
        "Unit root (non-stationary)",
        "Stationary",
        "No autocorrelation (white noise)",
        "No Granger causality",
        "No cointegration",
    ],
    "Reject H0 means": [
        "Series is stationary",
        "Series is non-stationary",
        "Autocorrelation exists",
        "X helps predict Y",
        "Series are cointegrated",
    ],
})
print(tests.to_string(index=False))

---
## Summary

| Test | Import | H0 | Reject when |
|------|--------|-----|-------------|
| Ljung-Box | `acorr_ljungbox` | No autocorrelation | p < 0.05 (autocorrelation exists) |
| Granger causality | `grangercausalitytests` | X does not predict Y | p < 0.05 (X helps predict Y) |
| Cointegration | `coint` | No cointegration | p < 0.05 (shared stochastic trend) |

**Key reminders:**
- Ljung-Box is primarily for checking model **residuals** — it answers "did
  the model capture all the autocorrelation?"
- Granger causality requires **stationary** data — difference first.
- Granger causality $\neq$ true causation.
- Cointegration is tested on **levels** (undifferenced data).

In [ ]:
print("Next: Notebook 04 — Time Series Features")
print("We will compute summary statistics and feature extraction across multiple series.")